# Road Damage Classification — M2 and M3
Run one cell at a time in a Colab GPU runtime. All code is visible here; no separate Python files are needed.

Use the existing M1 crops and split assignments. Compare:
1. A small CNN trained from scratch.
2. An ImageNet ResNet18 with a frozen backbone and a new classifier.
3. An ImageNet ResNet18 with all layers fine-tuned.

Choose checkpoints and the preferred model by **validation macro F1**. Keep the Norway test set for the final evaluation.

| Label | Damage |
|---|---|
| D00 | Longitudinal crack |
| D10 | Transverse crack |
| D20 | Alligator crack |
| D40 | Pothole |

This notebook needs real training results and your written interpretation before submission.


## 1. Set up Colab
Choose a GPU runtime. Install W&B and use Colab's existing PyTorch installation.


In [ ]:
%pip -q install wandb==0.19.8 scikit-learn


In [ ]:
from pathlib import Path
from importlib.metadata import version
import json
import random
import shutil
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from sklearn.metrics import classification_report, confusion_matrix, f1_score
import wandb

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", torch.cuda.get_device_name(0) if device.type == "cuda" else "CPU")


## 2. Connect Drive
The current project folder name ends with a space: **AI Project **. Keep that space in the path.
Teammates can change the paths and use a separate results folder.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")


In [ ]:
DATA_FOLDER = "/content/drive/MyDrive/AI Project /Data" #@param {type:"string"}
RESULTS_FOLDER = "/content/drive/MyDrive/AI Project /Results/m2_m3/alharby6626" #@param {type:"string"}
EXPERIMENT = "simple_v1" #@param {type:"string"}

data_dir = Path(DATA_FOLDER)
output_dir = Path(RESULTS_FOLDER) / EXPERIMENT
output_dir.mkdir(parents=True, exist_ok=True)
assert (data_dir / "classification_manifest.csv").is_file(), "Check DATA_FOLDER."
assert (data_dir / "classification_crops.zip").is_file(), "The crop ZIP is missing."
print("Data:", data_dir)
print("Results:", output_dir)


## 3. Read the ready-made M1 data
Use the split column from the existing classification manifest. These few checks catch missing labels,
duplicate crop paths and parent images appearing in more than one split. They do not recreate M1 or change its splits.


In [ ]:
classes = ["D00", "D10", "D20", "D40"]
class_to_id = {name: i for i, name in enumerate(classes)}
df = pd.read_csv(data_dir / "classification_manifest.csv")
assert df[["crop_path", "image_id", "class", "split"]].notna().all().all()
assert df["class"].isin(classes).all(), "Unknown class."
assert set(df["split"]) == {"train", "val", "test"}, "Missing or unknown split."
assert not df["crop_path"].duplicated().any(), "Duplicate crop path."
assert df.groupby("image_id")["split"].nunique().max() == 1, "Parent image crosses splits."
df["relative_path"] = df["crop_path"].str.replace("\\", "/", regex=False).str.split("classification_crops/").str[-1]
df["label"] = df["class"].map(class_to_id)
splits = {name: df[df["split"] == name].reset_index(drop=True) for name in ["train", "val", "test"]}
display(pd.crosstab(df["split"], df["class"]).reindex(["train", "val", "test"], columns=classes))


Unpack the existing ZIP onto Colab's local disk for faster image loading. This does not crop images again.
The archive can contain a classification_crops folder or start directly with train/val/test.


In [ ]:
local_zip = Path("/content/rdd_crops.zip")
crop_root = Path("/content/rdd_simple_crops")
shutil.copy2(data_dir / "classification_crops.zip", local_zip)

with zipfile.ZipFile(local_zip) as archive:
    for member in archive.infolist():
        if member.is_dir() or not member.filename.lower().endswith(".jpg"):
            continue
        parts = member.filename.replace("\\", "/").split("/")
        assert ".." not in parts, "Unsafe archive path."
        split, label, filename = parts[-3:]
        assert split in splits and label in classes, "Unexpected crop folder."
        target = crop_root / split / label / filename
        target.parent.mkdir(parents=True, exist_ok=True)
        with archive.open(member) as source, target.open("wb") as destination:
            shutil.copyfileobj(source, destination)

assert all((crop_root / p).is_file() for p in df["relative_path"]), "Some crops are missing."
print("Crops ready:", len(df))


## 4. Set training options
Use the same seed, data, image size, batch size and epoch count for all models.
Change EXPERIMENT before changing settings or restarting training.
Best checkpoints and histories are saved to Drive each epoch. This simple notebook does not resume
an interrupted optimizer. After reconnecting, rerun setup and skip training cells for completed models.


In [ ]:
SEED = 42
EPOCHS = 10
BATCH_SIZE = 32
IMAGE_SIZE = 224
LEARNING_RATES = {"scratch": 0.001, "frozen": 0.0003, "finetune": 0.0003}
WANDB_PROJECT = "rdd2022-road-damage"
WANDB_ENTITY = ""  # Team workspace name; blank uses your default workspace.
GROUP = "alharby6626_" + EXPERIMENT
model_names = ["scratch", "frozen", "finetune"]

def set_seed():
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed()
config = dict(seed=SEED, epochs=EPOCHS, batch_size=BATCH_SIZE, image_size=IMAGE_SIZE,
              optimizer="AdamW", weight_decay=0.0001, selection="val_macro_f1",
              classes=classes, data_folder=DATA_FOLDER, split="Unchanged M1 country split")
packages = ["torch", "torchvision", "numpy", "pandas", "Pillow", "matplotlib", "scikit-learn", "wandb"]
(output_dir / "requirements.txt").write_text("\n".join(f"{p}=={version(p)}" for p in packages))
print(config)


## 5. Load images
Apply horizontal flips only to training images. Both training and evaluation use 224-pixel resizing
and ImageNet normalization. Avoid random crops because M1 already cropped the damage regions.


In [ ]:
normalize = transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)), transforms.RandomHorizontalFlip(),
    transforms.ToTensor(), normalize
])
eval_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)), transforms.ToTensor(), normalize
])

class DamageDataset(Dataset):
    def __init__(self, rows, transform):
        self.rows, self.transform = rows.reset_index(drop=True), transform

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, index):
        row = self.rows.iloc[index]
        with Image.open(crop_root / row["relative_path"]) as image:
            image = self.transform(image.convert("RGB"))
        return image, int(row["label"])

def make_loader(split, training=False):
    dataset = DamageDataset(splits[split], train_transform if training else eval_transform)
    return DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=training, num_workers=2)

val_loader = make_loader("val")
print("Image loaders are ready.")


## 6. Define the models
The small CNN learns from scratch. Both ResNet18 experiments independently start from the same
ImageNet weights and replace the classifier with four outputs.
This practical comparison changes both architecture and initialization; it does not isolate pretraining alone.


In [ ]:
class SmallCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Dropout(0.3), nn.Linear(128, 4)
        )

    def forward(self, x):
        return self.network(x)

def build_model(name, pretrained=True):
    if name == "scratch":
        return SmallCNN().to(device)
    weights = models.ResNet18_Weights.IMAGENET1K_V1 if pretrained else None
    model = models.resnet18(weights=weights)
    if name == "frozen":
        for parameter in model.parameters():
            parameter.requires_grad = False
    model.fc = nn.Linear(model.fc.in_features, 4)
    return model.to(device)


## 7. Training and prediction
Reuse the same training functions for all three models. Frozen ResNet18 keeps its BatchNorm statistics fixed
and trains only its new classifier.


In [ ]:
loss_function = nn.CrossEntropyLoss()

def train_epoch(model, loader, optimizer, frozen=False):
    model.train()
    if frozen:
        model.eval()
        model.fc.train()
    total_loss, correct, count = 0, 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        logits = model(images)
        loss = loss_function(logits, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(labels)
        correct += (logits.argmax(1) == labels).sum().item()
        count += len(labels)
    return total_loss / count, correct / count


In [ ]:
@torch.no_grad()
def predict(model, loader):
    model.eval()
    labels_all, probabilities_all = [], []
    total_loss = 0
    for images, labels in loader:
        logits = model(images.to(device))
        total_loss += loss_function(logits, labels.to(device)).item() * len(labels)
        labels_all.append(labels.numpy())
        probabilities_all.append(logits.softmax(1).cpu().numpy())
    y = np.concatenate(labels_all)
    probabilities = np.concatenate(probabilities_all)
    return y, probabilities, total_loss / len(y)


## 8. Track training in W&B
Sign in at the prompt. Never put your API key in code or GitHub.
Check the workspace after login; team experiments should use the intended shared workspace.


In [ ]:
wandb.login()


In [ ]:
def train_model(name):
    set_seed()
    folder = output_dir / name
    assert not (folder / "best.pt").exists(), "Checkpoint exists. Skip this cell or use a new EXPERIMENT."
    folder.mkdir(parents=True, exist_ok=True)
    model = build_model(name)
    loader = make_loader("train", training=True)
    optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()),
                                 lr=LEARNING_RATES[name], weight_decay=0.0001)
    run_config = {**config, "model": name, "learning_rate": LEARNING_RATES[name]}
    (folder / "config.json").write_text(json.dumps(run_config, indent=2))
    history, best_f1 = [], -1
    with wandb.init(project=WANDB_PROJECT, entity=WANDB_ENTITY or None,
                    group=GROUP, name=name, job_type="training", config=run_config) as run:
        (folder / "wandb_url.txt").write_text(run.url or "")
        for epoch in range(1, EPOCHS + 1):
            train_loss, train_acc = train_epoch(model, loader, optimizer, name == "frozen")
            y, p, val_loss = predict(model, val_loader)
            val_f1 = f1_score(y, p.argmax(1), labels=range(4), average="macro", zero_division=0)
            row = dict(epoch=epoch, train_loss=train_loss, train_accuracy=train_acc,
                       val_loss=val_loss, val_accuracy=float((y == p.argmax(1)).mean()), val_macro_f1=val_f1)
            history.append(row)
            run.log(row)
            if val_f1 > best_f1:
                best_f1 = val_f1
                torch.save(model.state_dict(), folder / "best.pt")
                run.summary.update({"best_val_macro_f1": best_f1, "best_epoch": epoch})
            pd.DataFrame(history).to_csv(folder / "history.csv", index=False)
            print(f"{name} | epoch {epoch}/{EPOCHS} | val F1: {val_f1:.3f}")
    (folder / "training_complete.txt").write_text(f"Completed {EPOCHS} epochs")
    del model, optimizer
    torch.cuda.empty_cache()


## 9. Train the three models
Run each cell separately and wait until all its epochs finish.
ResNet18 downloads pretrained weights on its first use.


In [ ]:
train_model("scratch")


In [ ]:
train_model("frozen")


In [ ]:
train_model("finetune")


## 10. Plot learning curves
Training accuracy is measured on augmented images during updates; validation uses evaluation mode.


In [ ]:
for name in model_names:
    history = pd.read_csv(output_dir / name / "history.csv")
    fig, axes = plt.subplots(1, 2, figsize=(10, 3))
    history.plot(x="epoch", y=["train_loss", "val_loss"], ax=axes[0])
    history.plot(x="epoch", y=["train_accuracy", "val_accuracy"], ax=axes[1])
    axes[0].set_ylabel("Loss")
    axes[1].set_ylabel("Accuracy")
    fig.suptitle(name)
    fig.tight_layout()
    fig.savefig(output_dir / name / "learning_curves.png", dpi=150)
    plt.show()
    plt.close(fig)


## 11. Metrics, confusion matrix and calibration
Expected calibration error (ECE) compares confidence with observed accuracy in ten equal-width bins.
Lower ECE is better; a reliability curve close to the diagonal indicates better calibration.
These are the original softmax probabilities, without temperature scaling.


In [ ]:
def calibration_table(y, probabilities):
    confidence = probabilities.max(1)
    correct = probabilities.argmax(1) == y
    bins = np.minimum((confidence * 10).astype(int), 9)
    rows = []
    for b in range(10):
        mask = bins == b
        if mask.any():
            rows.append(dict(bin=b, count=int(mask.sum()),
                             confidence=float(confidence[mask].mean()), accuracy=float(correct[mask].mean())))
    table = pd.DataFrame(rows)
    ece = float((table["count"] * (table["accuracy"] - table["confidence"]).abs()).sum() / len(y))
    return table, ece


The test-accuracy confidence interval resamples parent images, keeping their crops together.
It describes uncertainty within the Norway sample, not across countries or training seeds.


In [ ]:
def accuracy_interval(rows, correct):
    grouped = pd.DataFrame({"image_id": rows["image_id"].to_numpy(), "correct": correct})
    counts = grouped.groupby("image_id")["correct"].agg(["sum", "count"]).to_numpy()
    rng = np.random.default_rng(SEED)
    scores = []
    for _ in range(1000):
        sample = counts[rng.integers(0, len(counts), len(counts))].sum(axis=0)
        scores.append(sample[0] / sample[1])
    return np.quantile(scores, [0.025, 0.975]).tolist()


In [ ]:
def save_plots(y, prediction, calibration, folder):
    matrix = confusion_matrix(y, prediction, labels=range(4))
    fig, ax = plt.subplots(figsize=(5, 4))
    ax.imshow(matrix, cmap="Blues")
    for i in range(4):
        for j in range(4):
            ax.text(j, i, str(matrix[i, j]), ha="center", va="center",
                    color="white" if matrix[i, j] > matrix.max() / 2 else "black")
    ax.set(xticks=range(4), yticks=range(4), xticklabels=classes, yticklabels=classes,
           xlabel="Predicted", ylabel="True", title="Confusion matrix")
    fig.tight_layout()
    fig.savefig(folder / "confusion_matrix.png", dpi=150)
    plt.close(fig)
    pd.DataFrame(matrix, index=classes, columns=classes).to_csv(folder / "confusion_matrix.csv")
    fig, ax = plt.subplots(figsize=(5, 4))
    ax.plot([0, 1], [0, 1], "--", label="Ideal")
    ax.plot(calibration["confidence"], calibration["accuracy"], "o-", label="Model")
    ax.set(xlim=(0, 1), ylim=(0, 1), xlabel="Mean confidence", ylabel="Accuracy", title="Reliability diagram")
    ax.legend()
    fig.tight_layout()
    fig.savefig(folder / "reliability.png", dpi=150)
    plt.close(fig)


In [ ]:
def evaluate_model(name, split):
    model_folder = output_dir / name
    assert (model_folder / "training_complete.txt").exists(), f"Finish training {name} first."
    model = build_model(name, pretrained=False)
    model.load_state_dict(torch.load(model_folder / "best.pt", map_location=device, weights_only=True))
    y, p, loss = predict(model, make_loader(split))
    prediction = p.argmax(1)
    folder = model_folder / split
    folder.mkdir(parents=True, exist_ok=True)
    report = pd.DataFrame(classification_report(y, prediction, labels=range(4), target_names=classes,
                                               output_dict=True, zero_division=0)).T
    report.to_csv(folder / "per_class_metrics.csv")
    calibration, ece = calibration_table(y, p)
    calibration.to_csv(folder / "calibration_bins.csv", index=False)
    save_plots(y, prediction, calibration, folder)
    metrics = dict(accuracy=float((prediction == y).mean()),
                   top2_accuracy=float((np.argsort(-p, axis=1, kind="stable")[:, :2] == y[:, None]).any(1).mean()),
                   macro_f1=float(report.loc["macro avg", "f1-score"]), ece=ece, loss=loss)
    if split == "test":
        metrics["accuracy_ci95_low"], metrics["accuracy_ci95_high"] = accuracy_interval(splits[split], prediction == y)
    (folder / "metrics.json").write_text(json.dumps(metrics, indent=2))
    predictions = splits[split].copy()
    predictions["predicted"] = np.array(classes)[prediction]
    predictions["confidence"] = p.max(1)
    predictions.to_csv(folder / "predictions.csv", index=False)
    del model
    torch.cuda.empty_cache()
    return metrics, report, predictions


In [ ]:
def log_evaluation(name, split, metrics, report):
    folder = output_dir / name / split
    with wandb.init(project=WANDB_PROJECT, entity=WANDB_ENTITY or None, group=GROUP,
                    name=f"{name}-{split}", job_type="evaluation",
                    config={**config, "model": name, "split": split}) as run:
        run.summary.update({f"{split}/{key}": value for key, value in metrics.items()})
        run.log({"per_class_metrics": wandb.Table(dataframe=report.loc[classes].reset_index()),
                 "confusion_matrix": wandb.Image(str(folder / "confusion_matrix.png")),
                 "reliability": wandb.Image(str(folder / "reliability.png")),
                 "learning_curves": wandb.Image(str(output_dir / name / "learning_curves.png"))})
        (folder / "wandb_url.txt").write_text(run.url or "")


## 12. Compare validation results
Macro F1 gives equal weight to each class. All experiments are compared in one W&B group.
Choose the highest validation macro F1; the first model wins an exact tie.
This step does not read test images.


In [ ]:
validation_rows = []
for name in model_names:
    metrics, report, predictions = evaluate_model(name, "val")
    log_evaluation(name, "val", metrics, report)
    validation_rows.append({"model": name, **metrics})
comparison = pd.DataFrame(validation_rows)
comparison.to_csv(output_dir / "validation_comparison.csv", index=False)
selected_model = comparison.loc[comparison["macro_f1"].idxmax(), "model"]
with wandb.init(project=WANDB_PROJECT, entity=WANDB_ENTITY or None,
                group=GROUP, name="validation-comparison", job_type="comparison") as run:
    run.log({"validation_comparison": wandb.Table(dataframe=comparison)})
    run.summary["selected_model"] = selected_model
display(comparison)
print("Selected using validation macro F1:", selected_model)


## 13. Inspect validation errors and write the improvement plan
View the selected model's most confident mistakes. The CSV contains all mistakes.
Record what you can actually see and link proposed actions to specific examples.
If there are no errors, review a random validation sample instead.


In [ ]:
predictions = pd.read_csv(output_dir / selected_model / "val" / "predictions.csv")
errors = predictions[predictions["class"] != predictions["predicted"]].sort_values("confidence", ascending=False)
errors.to_csv(output_dir / "validation_errors.csv", index=False)
display(errors.groupby(["class", "predicted"]).size().rename("count").sort_values(ascending=False).head(5))
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for ax in axes.flat:
    ax.axis("off")
for ax, (_, row) in zip(axes.flat, errors.head(8).iterrows()):
    with Image.open(crop_root / row["relative_path"]) as image:
        ax.imshow(image.convert("RGB"))
    ax.set_title(f"True: {row['class']} | Pred: {row['predicted']}\nConfidence: {row['confidence']:.2f}", fontsize=9)
fig.suptitle(f"Validation mistakes: {selected_model}")
fig.tight_layout()
fig.savefig(output_dir / "error_gallery.png", dpi=150)
plt.show()
plt.close(fig)


### Complete this table after reviewing your actual results
Replace the placeholders with observed mistakes and crop filenames.
Possible actions include adding training examples, reviewing ambiguous **training** labels, or improving training-image quality.
These are options, not established causes. Do not relabel validation/test data based on model predictions.

| Observed problem and example filename | Evidence from metrics/images | Proposed training-data improvement | Validation check |
|---|---|---|---|
| TODO | TODO | TODO | TODO |
| TODO | TODO | TODO | TODO |

Summarize the best model, main confusion pair, and whether confidence was reliable.
Discuss country shift: India/Japan/United States in training, China/Czech in validation, Norway in test.
Training is balanced but Norway is dominated by D00. M1's square crops may lose detail or distort shapes.
M1's visual-audit decisions need reconciliation with the original image IDs; do not quote an unverified label-error percentage.


## 14. Final test — only after finishing validation
Fix all model choices before setting the flag to True. Compare the three fixed best checkpoints on Norway.
The preferred model stays the one selected using validation. Do not tune or choose a different model using test results.


In [ ]:
FINAL_TEST_READY = False #@param {type:"boolean"}
if FINAL_TEST_READY:
    assert not (output_dir / "test_comparison.csv").exists(), "Final test already saved; read that CSV."
    test_rows = []
    for name in model_names:
        metrics, report, predictions = evaluate_model(name, "test")
        log_evaluation(name, "test", metrics, report)
        test_rows.append({"model": name, "selected_by_validation": name == selected_model, **metrics})
    test_comparison = pd.DataFrame(test_rows)
    test_comparison.to_csv(output_dir / "test_comparison.csv", index=False)
    with wandb.init(project=WANDB_PROJECT, entity=WANDB_ENTITY or None,
                    group=GROUP, name="test-comparison", job_type="comparison") as run:
        run.log({"test_comparison": wandb.Table(dataframe=test_comparison)})
        run.summary["selected_by_validation"] = selected_model
    display(test_comparison)
else:
    print("Test is closed. Complete validation review first.")


## Submission checklist
- Save this notebook with real outputs and your completed error analysis / data-improvement plan.
- Include the W&B group link, comparisons, learning curves and held-out Norway results.
- Include per-class precision/recall/F1, confusion matrices, reliability plots and ECE.
- Report top-1/top-2 accuracy, macro F1, the test-accuracy confidence interval and data limitations.
- Upload this notebook and the generated requirements.txt to your branch. Record the submitted commit in your report.
- Share the best checkpoints from Drive. Keep datasets, checkpoints and API keys out of GitHub.

Project training still needs to be run. One seed is a baseline, not a measure of variability across training runs.

References: [ResNet18](https://docs.pytorch.org/vision/stable/models/generated/torchvision.models.resnet18.html),
[classification metrics](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.classification_report.html),
[W&B logging](https://docs.wandb.ai/models/track/log).
